# Divar Fatemi Rental Anomaly Scanner v0.2
نسخهٔ مقاوم‌تر Colab: نصب و import را self-check می‌کند و سپس pipeline چندلایه را اجرا می‌کند.

> `risk_band` و `review_priority_score` فقط برای **اولویت‌بندی بازبینی** هستند؛ اثبات تقلب نیستند.

In [ ]:
import os, sys, shutil, subprocess, pathlib, importlib
REPO = '/content/divar_scanner_repo'  # عمداً با نام package یکی نیست
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git','clone','https://github.com/prestigegitserp/divar_scanner.git', REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

# Colab sometimes keeps an older/namespace module cached. Force the real src tree first.
SRC = str(pathlib.Path(REPO, 'src').resolve())
if SRC in sys.path: sys.path.remove(SRC)
sys.path.insert(0, SRC)
for name in list(sys.modules):
    if name == 'divar_scanner' or name.startswith('divar_scanner.'):
        del sys.modules[name]
importlib.invalidate_caches()

import divar_scanner, pandas as pd, numpy as np, sklearn, pyarrow
pkg = pathlib.Path(divar_scanner.__file__).resolve()
print('divar_scanner:', divar_scanner.__version__, pkg)
print('pandas:', pd.__version__, '| numpy:', np.__version__, '| sklearn:', sklearn.__version__, '| pyarrow:', pyarrow.__version__)
assert str(pkg).replace('\\','/').endswith('/src/divar_scanner/__init__.py'), f'WRONG PACKAGE ORIGIN: {pkg}'
print('✅ package/import self-check passed')


In [ ]:
# Environment doctor (network check is optional and does not crawl listings)
subprocess.run([sys.executable, '-m', 'divar_scanner.doctor', '--network'], check=True)


## تنظیمات
Backend معنایی اختیاری است. `none` یعنی فقط مدل‌های آماری/قاعده‌ای. برای اجرای کاملاً بدون پرداخت/کلید، `local-qwen` را انتخاب کن. `local-qwen-small` سبک‌تر است و `local-aya` فارسی را صریحاً پشتیبانی می‌کند.
کلید فقط در RAM این runtime قرار می‌گیرد و داخل GitHub یا فایل config ذخیره نمی‌شود.

In [ ]:
import getpass, yaml
from pathlib import Path

MAX_LISTINGS = 200  #@param {type:"integer"}
SEMANTIC_PROVIDER = 'none'  #@param ['none','local-qwen-small','local-qwen','local-aya','auto','groq','cerebras','gemini','cohere','openrouter','huggingface','jev','jev-official']\nENSEMBLE_MAX_PROVIDERS = 1  #@param {type:"integer"}
\nif SEMANTIC_PROVIDER.startswith('local-'):\n    print('Installing local inference extras...')\n    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[local]'], check=True)\n    import torch\n    print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')\n    if not torch.cuda.is_available() and SEMANTIC_PROVIDER != 'local-qwen-small':\n        raise RuntimeError('برای local-qwen/local-aya در Colab از Runtime > Change runtime type > GPU استفاده کن.')\n\n
env_for_provider = {
    'groq':'GROQ_API_KEY', 'cerebras':'CEREBRAS_API_KEY', 'gemini':'GEMINI_API_KEY',
    'cohere':'COHERE_API_KEY', 'openrouter':'OPENROUTER_API_KEY', 'huggingface':'HF_TOKEN', 'jev':'JEV_API_KEY',\n    'jev-official':'TYPESAFE_API_KEY'
}
if SEMANTIC_PROVIDER not in {'none','auto'} and not SEMANTIC_PROVIDER.startswith('local-'):
    env_name = env_for_provider[SEMANTIC_PROVIDER]
    key = getpass.getpass(f'{env_name} (hidden): ').strip()
    if key: os.environ[env_name] = key
    else: raise ValueError('No API key entered. Choose provider=none if you want a keyless run.')
os.environ['SEMANTIC_PROVIDER'] = SEMANTIC_PROVIDER if SEMANTIC_PROVIDER != 'none' else 'auto'

cfg = yaml.safe_load(Path('config/fatemi.yaml').read_text(encoding='utf-8'))
cfg['crawl']['max_listings'] = int(MAX_LISTINGS)
cfg['semantic']['enabled'] = SEMANTIC_PROVIDER != 'none'
cfg['semantic']['provider'] = 'auto' if SEMANTIC_PROVIDER in {'none','auto'} else SEMANTIC_PROVIDER
cfg['semantic']['ensemble_max_providers'] = max(1, int(ENSEMBLE_MAX_PROVIDERS))
runtime_cfg = Path('/content/fatemi_colab.yaml')
runtime_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('runtime config:', runtime_cfg)


In [ ]:
from divar_scanner.pipeline import run_pipeline
df, meta = run_pipeline('/content/fatemi_colab.yaml', crawl=True)
print('COUNTS:', meta.get('counts'))
print('SEMANTIC:', meta.get('semantic'))
print('OUTPUTS:', meta.get('outputs'))


In [ ]:
from IPython.display import display, HTML
cols = [
 'review_priority_score','suspicion_score','uncertainty_score','risk_band','title','neighborhood',
 'area_m2','rooms','deposit_toman','rent_monthly_toman','price_model_ratio',
 'price_model_anomaly_score','lof_anomaly_score','market_anomaly_score','data_quality_score',
 'duplicate_cluster_size','duplicate_bait_score','semantic_score','semantic_provider',
 'semantic_classification','semantic_confidence','flag_reasons','url'
]
display(df[[c for c in cols if c in df.columns]].head(50))

report = meta.get('outputs',{}).get('html_report')
if report and Path(report).exists():
    print('HTML report:', report)


In [ ]:
# Optional: download outputs from Colab
from google.colab import files
for label, path in meta.get('outputs', {}).items():
    print(label, '=>', path)
# Example: files.download(meta['outputs']['review_csv'])


## چرا این نسخه قوی‌تر است؟
- Robust peer statistics + Isolation Forest
- **Out-of-fold price model**: هر آگهی با مدلی score می‌شود که آن ردیف را در train ندیده است
- **LOF density anomaly** برای نقاط کم‌تراکم
- **Duplicate graph** برای خوشه‌های copy/paste و اختلاف محله/قیمت داخل خوشه
- Semantic backend قابل تعویض: Jev / Groq / Cerebras / Gemini / OpenRouter / Hugging Face / custom OpenAI-compatible
- uncertainty score برای مواردی که detectorها با هم اختلاف دارند- **Local open-weight inference**: Qwen3 1.7B/4B یا Aya Expanse 8B بدون API key\n